In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel
from transformers import pipeline
from sklearn.metrics import average_precision_score, ndcg_score




In [2]:
df = pd.read_csv("../data/vectors_hr_data.csv")
pd.set_option("display.max_colwidth", None)


In [3]:
measure_rank_1 = "aspiring human resources" 
measure_rank_2 =  "seeking human resources"

BoW  is defined as how many times does the word appear?
TF-IDF is defined as how important/distinctive is this word compared with the rest of the dataset?

Each word has a GloVe vector → average all word vectors → get one vector representing the entire job title.

Among the evaluated embedding approaches, E5 produced the most relevant job-title rankings. This is consistent with E5's retrieval-oriented training, which is specifically designed to represent semantically related texts close together in the embedding space.

In [4]:
def ranked_by_model(similarity_model, col, relevance):
    ranked = df.sort_values(similarity_model, ascending=False)

    return ranked[[col, similarity_model, relevance]]

In [7]:
df.columns

Index(['id', 'job_title', 'location', 'connection', 'relevance',
       'clean_job_title', 'similarity_1_bow', 'similarity_2_bow',
       'similarity_1_tfidf', 'similarity_2_tfidf', 'vector_for_modeling',
       'similarity_1_glove', 'similarity_2_glove', 'bert_for_modeling',
       'similarity_1_bert', 'similarity_2_bert', 'sbert_for_modeling',
       'similarity_1_sbert', 'similarity_2_sbert', 'e5_for_modeling',
       'similarity_1_5e', 'similarity_2_5e', 'bge_for_modeling',
       'similarity_1_bge', 'similarity_2_bge'],
      dtype='object')

In [9]:
comparsion_metric_1 = pd.DataFrame({
    "BoW": df.nlargest(10, "similarity_1_bow")["clean_job_title"].values,
    "TF-IDF": df.nlargest(10, "similarity_1_tfidf")["clean_job_title"].values,
    "GloVe": df.nlargest(10, "similarity_1_glove")["clean_job_title"].values,
    "BERT": df.nlargest(10, "similarity_1_bert")["clean_job_title"].values,
    "SBERT": df.nlargest(10, "similarity_1_sbert")["clean_job_title"].values,
    "E5": df.nlargest(10, "similarity_1_5e")["clean_job_title"].values,
    "BGE": df.nlargest(10, "similarity_1_bge")["clean_job_title"].values
})
comparsion_metric_1

,BoW,TF-IDF,GloVe,BERT,SBERT,E5,BGE
0,aspiring human resources manager seek internship human resources houston texas area,aspiring human resources manager seek internship human resources houston texas area,aspiring human resources manager seek internship human resources houston texas area,seek human resources opportunities chicago illinois,aspiring human resources professional raleigh durham north carolina area,aspiring human resources specialist greater new york city area,aspiring human resources professional raleigh durham north carolina area
1,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri,aspiring human resources specialist greater new york city area,human resources professional greater boston area,aspiring human resources manager seek internship human resources houston texas area,aspiring human resources specialist greater new york city area,aspiring human resources manager seek internship human resources houston texas area,aspiring human resources specialist greater new york city area
2,student humber college aspiring human resources generalist kanada,aspiring human resources professional raleigh durham north carolina area,seek human resources opportunities chicago illinois,aspiring human resources professional | energetic team focus leader austin texas area,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri,aspiring human resources professional raleigh durham north carolina area,aspiring human resources manager seek internship human resources houston texas area
3,aspiring human resources professional raleigh durham north carolina area,aspiring human resources management student seek internship houston texas area,aspiring human resources professional | passionate help create inclusive engage work environment new york new york,student humber college aspiring human resources generalist kanada,aspiring human resources professional | energetic team focus leader austin texas area,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri,aspiring human resources professional | passionate help create inclusive engage work environment new york new york
4,aspiring human resources specialist greater new york city area,student humber college aspiring human resources generalist kanada,director human resources ey greater atlanta area,experienced retail manager aspire human resources professional austin texas area,aspiring human resources manager seek internship human resources houston texas area,aspiring human resources professional | passionate help create inclusive engage work environment new york new york,aspiring human resources professional | energetic team focus leader austin texas area
5,aspiring human resources management student seek internship houston texas area,business management major aspiring human resources manager monroe louisiana area,aspiring human resources specialist greater new york city area,business management major aspiring human resources manager monroe louisiana area,aspiring human resources management student seek internship houston texas area,aspiring human resources management student seek internship houston texas area,aspiring human resources management student seek internship houston texas area
6,business management major aspiring human resources manager monroe louisiana area,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri,seek human resources hris generalist positions greater philadelphia area,aspiring human resources management student seek internship houston texas area,seek human resources position las vegas nevada area,aspiring human resources professional | energetic team focus leader austin texas area,aspiring human resources manager | graduating 2020 | seek entry level human resources posit

| **Precision@5** | Fraction of the first 5 candidates that are relevant.
| **Precision@10** | Fraction of the first 10 candidates that are relevant. 
| **MRR** | How early the first relevant candidate appears. 
| **AP** | Looks across the entire ranking and rewards putting relevant candidates before irrelevant ones. 
| **NDCG@10** | actual ranking score / best possible ranking score

In [10]:
comparsion_metric_2 = pd.DataFrame({
    "BoW": df.nlargest(10, "similarity_2_bow")["clean_job_title"].values,
    "TF-IDF": df.nlargest(10, "similarity_2_tfidf")["clean_job_title"].values,
    "GloVe": df.nlargest(10, "similarity_2_glove")["clean_job_title"].values,
    "BERT": df.nlargest(10, "similarity_2_bert")["clean_job_title"].values,
    "SBERT": df.nlargest(10, "similarity_2_sbert")["clean_job_title"].values,
    "E5": df.nlargest(10, "similarity_2_5e")["clean_job_title"].values,
    "BGE": df.nlargest(10, "similarity_2_bge")["clean_job_title"].values
})
comparsion_metric_2

,BoW,TF-IDF,GloVe,BERT,SBERT,E5,BGE
0,aspiring human resources manager seek internship human resources houston texas area,aspiring human resources manager seek internship human resources houston texas area,aspiring human resources manager seek internship human resources houston texas area,seek human resources opportunities chicago illinois,seek human resources position las vegas nevada area,seek human resources opportunities chicago illinois,seek human resources opportunities chicago illinois
1,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri,human resources professional greater boston area,seek human resources opportunities chicago illinois,human resources staffing recruiting professional jackson mississippi area,aspiring human resources professional raleigh durham north carolina area,seek human resources position las vegas nevada area,seek human resources position las vegas nevada area
2,seek human resources opportunities chicago illinois,human resources management major milpitas california,seek human resources hris generalist positions greater philadelphia area,aspiring human resources manager seek internship human resources houston texas area,aspiring human resources specialist greater new york city area,aspiring human resources manager seek internship human resources houston texas area,seek human resources hris generalist positions greater philadelphia area
3,human resources professional greater boston area,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri,seek human resources opportunity open travel relocation amerika birlesik devletleri,director administration excellence logging katy texas,seek human resources opportunities chicago illinois,seek human resources hris generalist positions greater philadelphia area,aspiring human resources professional raleigh durham north carolina area
4,human resources management major milpitas california,director human resources ey greater atlanta area,human resources professional greater boston area,seek human resources hris generalist positions greater philadelphia area,human resources generalist loparex raleigh durham north carolina area,seek human resources opportunity open travel relocation amerika birlesik devletleri,aspiring human resources manager seek internship human resources houston texas area
5,human resources generalist schwan amerika birlesik devletleri,seek human resources opportunities chicago illinois,director human resources ey greater atlanta area,seek employment opportunity customer service patient care torrance california,seek human resources hris generalist positions greater philadelphia area,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri,aspiring human resources specialist greater new york city area
6,director human resources ey greater atlanta area,aspiring human resources specialist greater new york city area,human resources management major milpitas california,people development coordinator ryan denton texas,human resources professional greater boston area,aspiring human resources management student seek internship houston texas area,human resources staffing recruiting professional jackson mississippi area
7,student humber college aspiring human resources generalist kanada,aspiring human resources professional raleigh durham north carolina area,aspiring human resources management student seek internship houston texas area,director human resources ey greater atlanta area,aspiring human resources manager seek internship human resources houston texas area,aspiring human resources specialist greater new york city area,seek human resources opportunity open travel relocation amerika birlesik devletleri
8,human resources staffing recruiting professional jackson mississippi area,aspiring human resources management student seek inte

In [6]:
def metrics(ranked, relevance, similarity_model):
    labels = ranked[relevance].to_numpy()
    scores = ranked[similarity_model].to_numpy()

    precision_5 = labels[:5].sum() / 5
    precision_10 = labels[:10].sum() / 10

    relevant_positions = np.where(labels == 1)[0] + 1
    mrr = 1 / relevant_positions[0] if len(relevant_positions) else 0

    if labels.sum() > 0:
        ap = average_precision_score(labels, scores)
        ndcg_10 = ndcg_score(
            labels.reshape(1, -1),
            scores.reshape(1, -1),
            k=10
        )
    else:
        ap = 0
        ndcg_10 = 0

    return {
        "Precision@5": precision_5,
        "Precision@10": precision_10,
        "MRR": mrr,
        "AP": ap,
        "NDCG@10": ndcg_10
    }


In [7]:

bow_ranked = ranked_by_model("similarity_1_bow","clean_job_title","relevance")
bow_metrics = metrics(bow_ranked, "relevance","similarity_1_bow")

tfidf_ranked = ranked_by_model("similarity_1_tfidf","clean_job_title","relevance")
tfidf_metrics = metrics(tfidf_ranked, "relevance","similarity_1_tfidf")

glove_ranked = ranked_by_model("similarity_1_glove","clean_job_title","relevance")
glove_metrics = metrics(glove_ranked, "relevance","similarity_1_glove")

bert_ranked = ranked_by_model("similarity_1_bert","clean_job_title","relevance")
bert_metrics = metrics(bert_ranked, "relevance","similarity_1_bert")

sbert_ranked = ranked_by_model("similarity_1_sbert","clean_job_title","relevance")
sbert_metrics = metrics(sbert_ranked, "relevance","similarity_1_sbert")

e5_ranked = ranked_by_model("similarity_1_5e","clean_job_title","relevance")
e5_metrics = metrics(e5_ranked, "relevance","similarity_1_5e")

bge_ranked = ranked_by_model("similarity_1_bge","clean_job_title","relevance")
bge_metrics = metrics(bge_ranked, "relevance","similarity_1_bge")

In [8]:
bow_ranked = ranked_by_model("similarity_2_bow","clean_job_title","relevance")
bow_metrics = metrics(bow_ranked, "relevance","similarity_2_bow")

tfidf_ranked = ranked_by_model("similarity_2_tfidf","clean_job_title","relevance")
tfidf_metrics = metrics(tfidf_ranked, "relevance","similarity_2_tfidf")

glove_ranked = ranked_by_model("similarity_2_glove","clean_job_title","relevance")
glove_metrics = metrics(glove_ranked, "relevance","similarity_2_glove")

bert_ranked = ranked_by_model("similarity_2_bert","clean_job_title","relevance")
bert_metrics = metrics(bert_ranked, "relevance","similarity_2_bert")

sbert_ranked = ranked_by_model("similarity_2_sbert","clean_job_title","relevance")
sbert_metrics = metrics(sbert_ranked, "relevance","similarity_2_sbert")

e5_ranked = ranked_by_model("similarity_2_5e","clean_job_title","relevance")
e5_metrics = metrics(e5_ranked, "relevance","similarity_2_5e")

bge_ranked = ranked_by_model("similarity_2_bge","clean_job_title","relevance")
bge_metrics = metrics(bge_ranked, "relevance","similarity_2_bge")

In [ ]:
results = pd.DataFrame({
    "Bow": bow_metrics,
    "TF-IDF": tfidf_metrics,
    "GloVe": glove_metrics,
    "BERT": bert_metrics,
    "SBERT": sbert_metrics,
    "E5": e5_metrics,
    "BGE": bge_metrics
}).T.rename_axis("Model").reset_index()

results

In [9]:
results = pd.DataFrame({
    "Bow": bow_metrics,
    "TF-IDF": tfidf_metrics,
    "GloVe": glove_metrics,
    "BERT": bert_metrics,
    "SBERT": sbert_metrics,
    "E5": e5_metrics,
    "BGE": bge_metrics
}).T.rename_axis("Model").reset_index()

results

,Model,Precision@5,Precision@10,MRR,AP,NDCG@10
0,Bow,1.0,1.0,1.0,0.977855,1.000000
1,TF-IDF,1.0,1.0,1.0,0.977855,1.000000
2,GloVe,1.0,1.0,1.0,0.955214,1.000000
3,BERT,0.8,0.7,1.0,0.831193,0.753449
4,SBERT,1.0,1.0,1.0,0.995338,1.000000
5,E5,1.0,1.0,1.0,0.990001,1.000000
6,BGE,1.0,1.0,1.0,0.987924,1.000000


In [17]:
tokenizer = AutoTokenizer.from_pretrained(
    "intfloat/e5-small-v2"
)

e5 = AutoModel.from_pretrained(
    "intfloat/e5-small-v2"
)


def get_title_vector_e5(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = e5(**inputs)

    # Token embeddings
    token_embeddings = outputs.last_hidden_state

    # Attention mask
    attention_mask = inputs["attention_mask"]

    # Expand mask to same dimensions as embeddings
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    # Mean pooling
    vector = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

    return vector.squeeze().numpy()


df["e5_for_modeling"] = df["clean_job_title"].apply(
    get_title_vector_e5
)




X_e5 = np.vstack(df["e5_for_modeling"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [19]:
y1_vector = get_title_vector_e5(measure_rank_1)
y2_vector = get_title_vector_e5(measure_rank_2)

score_1 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_e5
)

score_2 = cosine_similarity(
    y1_vector.reshape(1, -1),
    X_e5
)

In [21]:
star_candidate = X_e5[6] # student humber college aspiring human resource

starred_similarity = cosine_similarity(
    X_e5,
    star_candidate.reshape(1, -1)
).flatten()

alpha = 0.6

df['rerank_score'] = (
    alpha * score_2.flatten()
    + (1 - alpha) * starred_similarity
)

df[["clean_job_title", "rerank_score"]].sort_values(
    "rerank_score",
    ascending=False
).head(10)

,clean_job_title,rerank_score
6,student humber college aspiring human resources generalist kanada,0.909731
5,aspiring human resources specialist greater new york city area,0.874429
21,aspiring human resources manager seek internship human resources houston texas area,0.872948
2,aspiring human resources professional raleigh durham north carolina area,0.871230
12,aspiring human resources management student seek internship houston texas area,0.869178
47,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri,0.867184
24,aspiring human resources professional | passionate help create inclusive engage work environment new york new york,0.856941
30,aspiring human resources professional | energetic team focus leader austin texas area,0.853395
20,business management major aspiring human resources manager monroe louisiana area,0.848845
13,seek human resources opportunities chicago illinois,0.847606


In [22]:
star_candidates = X_e5[2:6] # student number college aspiring human resource
star_anchor = star_candidates.mean(axis=0)
starred_similarity = cosine_similarity(
    X_e5,
    star_anchor.reshape(1, -1)
).flatten()

alpha = 0.5

df['rerank_scores'] = (
    alpha * score_2.flatten()
    + (1 - alpha) * starred_similarity
)

df[["clean_job_title", "rerank_scores"]].sort_values(
    "rerank_scores",
    ascending=False
).head(10)

,clean_job_title,rerank_scores
5,aspiring human resources specialist greater new york city area,0.918579
2,aspiring human resources professional raleigh durham north carolina area,0.916666
21,aspiring human resources manager seek internship human resources houston texas area,0.900285
30,aspiring human resources professional | energetic team focus leader austin texas area,0.897132
24,aspiring human resources professional | passionate help create inclusive engage work environment new york new york,0.892534
12,aspiring human resources management student seek internship houston texas area,0.890986
47,aspiring human resources manager | graduating 2020 | seek entry level human resources position st. louis cape girardeau missouri,0.887913
22,human resources professional greater boston area,0.878556
13,seek human resources opportunities chicago illinois,0.875295
46,seek human resources position las vegas nevada area,0.874984


In [11]:
def zero_shot_classifer_check():
    zero_shot_classifier = pipeline(
        task="zero-shot-classification",
        model="facebook/bart-large-mnli"
    )

    candidate_labels = [
        "This candidate HR related",
        "This candidate not HR related"
    ]

    texts = df["clean_job_title"].tolist()

    results = zero_shot_classifier(
        texts,
        candidate_labels=candidate_labels
    )

    return results

In [12]:
results = zero_shot_classifer_check()
df_results = pd.DataFrame(results)
df_results

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

,sequence,labels,scores
0,2019 c.t. bauer college business graduate magna cum laude aspire human resources professional houston texas,"[This candidate HR related, This candidate not HR related]","[0.9920650720596313, 0.007934912107884884]"
1,native english teacher epik english program korea kanada,"[This candidate HR related, This candidate not HR related]","[0.5143134593963623, 0.48568660020828247]"
2,aspiring human resources professional raleigh durham north carolina area,"[This candidate HR related, This candidate not HR related]","[0.992178738117218, 0.007821290753781796]"
3,people development coordinator ryan denton texas,"[This candidate HR related, This candidate not HR related]","[0.9490283131599426, 0.05097171664237976]"
4,advisory board member celal bayar university izmir turkiye,"[This candidate HR related, This candidate not HR related]","[0.7052016854286194, 0.2947983145713806]"
5,aspiring human resources specialist greater new york city area,"[This candidate HR related, This candidate not HR related]","[0.9920141696929932, 0.007985870353877544]"
6,student humber college aspiring human resources generalist kanada,"[This candidate HR related, This candidate not HR related]","[0.9924695491790771, 0.007530476897954941]"
7,hr senior specialist san francisco bay area,"[This candidate HR related, This candidate not HR related]","[0.8218913674354553, 0.17810866236686707]"
8,seek human resources hris generalist positions greater philadelphia area,"[This candidate HR related, This candidate not HR related]","[0.9885026216506958, 0.011497434228658676]"
9,student chapman university lake forest california,"[This candidate not HR related, This candidate HR related]","[0.8107824325561523, 0.18921761214733124]"


I examined the distribution of candidate locations using a count plot. Some locations were more represented than others, but this does not demonstrate ranking bias. Because location and connection count were included in the combined text used for embeddings, they may influence similarity scores. Further evaluation would compare rankings generated from job titles alone with rankings generated from the combined text.